# Image-to-Video server (LTX-Video + Gradio public link)
1. Runtime > Change runtime type > **GPU** (T4 free works but is slow; L4/A100 is faster).
2. Run cells 1 -> 3 in order. Cell 3 prints a **public gradio.live URL**: copy it and send it to Claude.
3. Keep this tab open while clips are generating (Colab stops idle sessions).

Each call makes one ~5 s clip (121 frames @ 24 fps, 704x480) from an input image + prompt and
also returns the **last frame**, so the next clip can start where this one ended (continuity).

In [ ]:
!pip install -q -U "diffusers>=0.32" transformers accelerate sentencepiece imageio imageio-ffmpeg gradio gradio_client

In [ ]:
import torch
from diffusers import LTXImageToVideoPipeline
from diffusers.utils import export_to_video

assert torch.cuda.is_available(), "Turn on a GPU runtime first"
pipe = LTXImageToVideoPipeline.from_pretrained("Lightricks/LTX-Video", torch_dtype=torch.bfloat16)
pipe.enable_model_cpu_offload()      # fits a 16 GB T4
pipe.vae.enable_tiling()
print("model ready")

In [ ]:
import gradio as gr, tempfile
from PIL import Image

NEG = "worst quality, inconsistent motion, blurry, jittery, distorted, 3d render, photorealistic, watermark, text"

def generate(image, prompt, num_frames=121, steps=40, seed=0):
    img = image.convert("RGB").resize((704, 480))
    gen = torch.Generator("cuda").manual_seed(int(seed))
    frames = pipe(image=img, prompt=prompt, negative_prompt=NEG, width=704, height=480,
                  num_frames=int(num_frames), num_inference_steps=int(steps),
                  guidance_scale=3.0, generator=gen).frames[0]
    vid = tempfile.mktemp(suffix=".mp4"); export_to_video(frames, vid, fps=24)
    last = tempfile.mktemp(suffix=".png"); frames[-1].save(last)
    return vid, last

with gr.Blocks() as demo:
    gr.Markdown("### Image -> Video (LTX-Video)")
    with gr.Row():
        with gr.Column():
            im = gr.Image(type="pil", label="Start frame")
            pr = gr.Textbox(label="Prompt", lines=4)
            nf = gr.Slider(25, 161, value=121, step=8, label="Frames (8k+1: 121 = 5 s)")
            st = gr.Slider(10, 50, value=40, step=1, label="Steps")
            sd = gr.Number(value=0, label="Seed", precision=0)
            go = gr.Button("Generate")
        with gr.Column():
            ov = gr.Video(label="Clip")
            ol = gr.Image(type="filepath", label="Last frame (use as next start frame)")
    go.click(generate, [im, pr, nf, st, sd], [ov, ol], api_name="generate")

demo.queue(max_size=16).launch(share=True, debug=True)   # prints the public URL